# 2.2 — Feature Engineering (Lag)

Tahap preprocessing bagian 2: membuat fitur lag (3 bulan dan 6 bulan) untuk semua variabel independen berbasis waktu.

**Input:** `2_data_preprocessing/output/2.1_cleaned_smoothed_data.csv`

**Output:** `2_data_preprocessing/output/2.2_final_feature_set.csv`

In [ ]:
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / '2_data_preprocessing' / 'output' / '2.1_cleaned_smoothed_data.csv').exists():
            return p
        if (p / '1_data_gathering' / 'output' / '1_raw_panel_data.csv').exists():
            return p
    raise FileNotFoundError('Could not find the expected pipeline CSVs in current or parent directories.')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / '2_data_preprocessing' / 'output' / '2.1_cleaned_smoothed_data.csv'
output_dir = ROOT / '2_data_preprocessing' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / '2.2_final_feature_set.csv'

df = pd.read_csv(input_path)
df['tanggal'] = pd.to_datetime(df['tanggal'])
df = df.sort_values(by=['provinsi_id', 'tanggal']).reset_index(drop=True)

print(f'Loaded: {input_path}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} cols')

In [ ]:
# Pilih fitur dasar (tanpa kolom lag) yang dimulai dengan 'x'
base_x_features = [col for col in df.columns if col.startswith('x') and '_lag_' not in col]
print(f'Total base features: {len(base_x_features)}')

# Isi missing values residual pada fitur dasar per provinsi
for feature in base_x_features:
    df[feature] = df.groupby('provinsi_id')[feature].transform(lambda s: s.ffill().bfill())
    if df[feature].isna().any():
        df[feature] = df[feature].fillna(df[feature].median())

# Buat fitur lag 3 bulan dan 6 bulan
lags = [3, 6]
for lag in lags:
    for feature in base_x_features:
        df[f'{feature}_lag_{lag}'] = df.groupby('provinsi_id')[feature].shift(lag)

lag_cols = [col for col in df.columns if '_lag_' in col]
print('--- Missing Values Pasca-Lagging (Top 10) ---')
display(df[lag_cols].isna().sum().sort_values(ascending=False).head(10))

assert (df['provinsi_id'] == 19).sum() == 0, 'provinsi_id 19 still present (expected removed upstream).'

df.to_csv(output_path, index=False)
print(f'Saved: {output_path}')